# Downloading DPA RegioLine press releases

As a possible better alternative to news articles. For example, they can be expected to be de-duplicated already, and they should have a balanced geographical distribution, and only refer to events within Germany, and clearly state the location.

In [1]:
from protest_impact.data.protests.config import search_string

search_string

'protest* OR demo OR demonstr* OR kundgebung OR versamm* OR "soziale bewegung" OR hausbesetz* OR streik* OR unterschriften* OR petition OR hasskriminalität OR unruhen OR aufruhr OR aufstand OR rebell* OR blockade OR blockier* OR sitzblock* OR boykott* OR riot OR aktivis* OR bürgerinitiative OR bürgerbegehren OR marsch OR aufmarsch OR parade OR mahnwache OR hungerstreik OR "ziviler ungehorsam"'

In [1]:
from calendar import monthrange
from os import listdir
from time import sleep

from pyperclip import paste

download_dir = "/Users/david/Downloads"
n_files = len(listdir(download_dir))
print(n_files)

427


In [36]:
from subprocess import run


def click(args: str):
    run(["cliclick", "-e 1000", *args.split(" ")])


def move(x: int, y: int):
    click(f"m:{x},{y}")

In [ ]:
sleep(1)
click("kd:cmd kp:tab ku:cmd")
sleep(1)

year = 2017
month = 9
i = 2
while year > 2012:
    sleep(5)
    while month > 0:
        sleep(5)
        move(362, 373)
        click("c:.")
        sleep(10)
        move(250, 670)
        click("c:.")
        sleep(1)
        click("kd:cmd t:a ku:cmd")
        sleep(1)
        click(f"t:01/{str(month).zfill(2)}/{year}")
        sleep(1)
        move(250, 700)
        click("c:.")
        sleep(1)
        click("kd:cmd t:a ku:cmd")
        sleep(1)
        last_day = monthrange(year, month)[1]
        click(f"t:{str(last_day).zfill(2)}/{str(month).zfill(2)}/{year}")
        sleep(1)
        move(330, 735)
        click("c:.")
        sleep(7)
        move(490, 203)
        click("dc:.")
        sleep(1)
        click("kd:cmd t:c ku:cmd")
        sleep(1)
        clipboard = paste()
        print("📋", clipboard)
        max_items = int(clipboard)
        while i <= (max_items - 1) // 100:
            sleep(5)
            move(622, 240)
            click("c:.")
            sleep(1)
            move(500, 305)
            click("c:.")
            sleep(1)
            end = min((i + 1) * 100, max_items)
            if i * 100 + 1 == end:
                click(f"ku:cmd t:{str(i*100+1)}")
            else:
                click(f"ku:cmd t:{str(i*100+1)}-{str(end)}")
            sleep(1)
            # move(500, 720)
            # click("dc:.")
            # sleep(1)
            # click(f'ku:cmd t:dpa-regio-2013-2022_{str(year)}-{str(month).zfill(2)}')
            # sleep(1)
            n_files = len(listdir(download_dir))
            move(900, 810)
            click("c:.")  # Download
            sleep(5)
            while len(listdir(download_dir)) == n_files:
                sleep(10)
            print(year, month, i)
            sleep(5)
            i += 1
        i = 0
        month -= 1
    month = 12
    year -= 1

click("kd:cmd kp:tab ku:cmd")

In [3]:
import json
import re
from pathlib import Path
from zipfile import ZipFile

from dateparser import parse
from striprtf.striprtf import rtf_to_text
from tqdm.notebook import tqdm

from protest_impact.util import project_root

items = []
fails = []
for i, file in tqdm(list(enumerate(Path(download_dir).glob("Files*.ZIP")))):
    with ZipFile(file) as zipObj:
        for file in zipObj.filelist:
            rtf = zipObj.read(file).decode("utf-8")
            plaintext = rtf_to_text(rtf, errors="ignore").strip()
            plaintext = plaintext.replace("\xa0", " ")
            try:
                title, rest = plaintext.split("dpa RegioLine", 1)
                date, rest = rest.strip().split("\n", 1)
                date = parse(date.strip())
                meta, rest = rest.split("Body", 1)
                if rest.strip().startswith("Zusammenfassung\n"):
                    _, summary, rest = rest.strip().split("\n", 2)
                else:
                    summary = ""
                location, rest = re.split(r" ?\(dpa| ?\(dap", rest, 1)
                if rest.startswith("/"):
                    region, rest = re.split(r"\) ?- ?|\) ?– ?", rest[1:], 1)
                else:
                    region = ""
                    rest = rest[4:]
                if "Graphic" in rest:
                    text, graphic = rest.split("Graphic", 1)
                else:
                    text, rest = rest.split("Load-Date", 1)
                items.append(
                    {
                        "title": title.strip(),
                        "date": date.isoformat(),
                        "summary": summary.strip(),
                        "location": location.strip(),
                        "region": region.strip(),
                        "text": text.strip(),
                    }
                )
            except Exception as e:
                fails.append(plaintext)
    if i % 30 == 0:
        with open(project_root / f"data/news/dpa_{i}.jsonl", "w") as f:
            for item in items:
                f.write(json.dumps(item) + "\n")

with open(project_root / "data/news/dpa.jsonl", "w") as f:
    for item in items:
        f.write(json.dumps(item) + "\n")

len(items), len(fails)

  0%|          | 0/379 [00:00<?, ?it/s]

(33060, 557)